# Paderborn University Dataset - 3상 전류 STFT 예비 실습
이 노트북은 AMPI 센서리스 플랫폼을 위해 **Paderborn University Bearing Dataset**을 활용하는 방법을 안내하고 실습하는 코드입니다.

## 0. 사전 준비 (데이터 다운로드)
Paderborn 대학 데이터셋은 용량이 매우 크므로 수동 다운로드가 필요합니다.
1. **다운로드 링크:** [KAt DataCenter](https://mb.uni-paderborn.de/kat/forschung/bearing-datacenter) (또는 Kaggle에서 `christianlillelund/paderborn-university-bearing-dataset` 검색)
2. 다운로드 받은 `.mat` 파일(예: 정상 데이터인 `K001.mat`)을 이 노트북과 같은 폴더에 위치시켜주세요.

Paderborn 데이터는 진동과 **2상 전류(Phase 1, 2)**를 64kHz(64000Hz)로 측정했습니다.\n

In [ ]:
# 필요 라이브러리 설치 (최초 1회 실행)
!pip install scipy matplotlib pandas numpy
\n

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
import scipy.io as sio
import os\n

## 1. 2상 전류 로드 및 3상 전류(Phase 3) 복원
Paderborn 데이터셋은 Phase 1, Phase 2의 전류만 측정되었습니다. 
하지만 3상 평형 시스템(Y결선 또는 $\Delta$결선 모터)에서는 키르히호프의 전류 법칙(KCL)에 의해 **$I_1 + I_2 + I_3 = 0$**이 성립합니다.
따라서 측정되지 않은 3번째 상의 전류는 **$I_3 = -(I_1 + I_2)$** 로 완벽하게 복원할 수 있습니다.\n

In [ ]:
# 예시 파일명 (다운로드 받은 mat 파일 이름으로 변경하세요)
mat_file_path = 'K001.mat'

# 샘플링 주파수 (Paderborn 전류 센서는 64kHz)
Fs = 64000 

# 파일이 존재하는 경우 실제 데이터를 불러오고, 없으면 실습용 시뮬레이션 데이터를 생성합니다.
if os.path.exists(mat_file_path):
    print("실제 Paderborn .mat 파일을 불러옵니다.")
    mat_data = sio.loadmat(mat_file_path)
    
    # mat 구조 탐색 (구조체 내부의 전류 데이터 접근)
    # 일반적으로 mat_data['K001'][0][0]['Y'][0][0]['Data'][0] 등이 전류 1, [1]이 전류 2, [2]가 진동 데이터입니다.
    # 복잡성을 피하기 위해 여기서는 직접 불러온 데이터에서 전류를 추출했다고 가정합니다.
    # (주의: 실제 Paderborn .mat 구조에 따라 인덱스가 조금 다를 수 있습니다)
    
    phase1_current = mat_data['K001'][0][0]['Y'][0][0]['Data'][0][0][:64000] # 1초 분량
    phase2_current = mat_data['K001'][0][0]['Y'][0][0]['Data'][1][0][:64000]
    
else:
    print(f"{mat_file_path} 파일을 찾을 수 없어, Paderborn 데이터 특성과 동일한 시뮬레이션 데이터를 생성합니다.")
    t_sim = np.linspace(0, 1, Fs, endpoint=False) # 1초 데이터
    
    # 정상 상태의 2상 전류 (60Hz 가정, 120도 위상차) + 노이즈
    phase1_current = np.sin(2 * np.pi * 60 * t_sim) + np.random.normal(0, 0.05, Fs)
    phase2_current = np.sin(2 * np.pi * 60 * t_sim - 2*np.pi/3) + np.random.normal(0, 0.05, Fs)
    
    # 만약 고장 데이터라면 고주파 Harmonics 추가
    # phase1_current += 0.02 * np.sin(2 * np.pi * 1500 * t_sim)
    # phase2_current += 0.02 * np.sin(2 * np.pi * 1500 * t_sim)

# 핵심 원리: Phase 3 전류 복원!
phase3_current = -(phase1_current + phase2_current)

# 3상 전류 배열로 통합 (Shape: N x 3)
data_3ch = np.column_stack([phase1_current, phase2_current, phase3_current])

print("데이터 Shape (1초 분량, 3상 전류):", data_3ch.shape)


## 2. STFT 계산 및 AI(CNN)용 전처리 파이프라인
이제 완벽한 3상 전류 데이터가 준비되었으므로, AMPI 발표자료의 의도대로 **(Freq, Time, 3 Channels)** 형태의 **RGB 이미지 배열**로 융합합니다.\n

In [ ]:
def preprocess_stft_for_cnn(sensor_data_3ch, fs, nperseg=256, noverlap=128):
    """
    3채널 3상 전류 데이터를 받아 CNN 입력용 (H, W, 3) 이미지 텐서로 변환합니다.
    """
    channels_stft = []
    
    for i in range(3):
        # 1. STFT 변환 (전류 -> 주파수/시간 도메인)
        f, t, Zxx = signal.stft(sensor_data_3ch[:, i], fs=fs, nperseg=nperseg, noverlap=noverlap)
        
        mag = np.abs(Zxx)
        
        # 2. Log-scale 변환 (dB 스케일) : 미세한 고주파 고장 패턴(Harmonics)을 딥러닝 모델이 잘 포착하도록 증폭
        mag_db = 20 * np.log10(mag + 1e-10)
        
        # 3. Min-Max 정규화 (0 ~ 1) : CNN 학습 필수 조건
        mag_norm = (mag_db - np.min(mag_db)) / (np.max(mag_db) - np.min(mag_db))
        
        channels_stft.append(mag_norm)
    
    # 3상 STFT 결과를 (R, G, B) 채널처럼 스태킹 (Axis=-1)
    stft_image = np.stack(channels_stft, axis=-1)
    
    return f, t, stft_image

# nperseg=2048 로 설정 (64kHz 샘플링에서는 nperseg가 커야 주파수 해상도(약 31.25Hz)가 나옵니다)
f, t, stft_image = preprocess_stft_for_cnn(data_3ch, Fs, nperseg=2048, noverlap=1024)

print("최종 CNN 입력용 STFT Tensor Shape (Freq, Time, Channels):", stft_image.shape)
\n

## 3. 결과 시각화 (전류의 이미지화)
CNN 모델에 투입될 전처리된 배열을 시각화합니다. 
* Phase 1은 Red 채널, Phase 2는 Green 채널, 복원된 Phase 3은 Blue 채널에 대응한다고 볼 수 있습니다.\n

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)
channel_names = ['Phase 1 Current', 'Phase 2 Current', 'Phase 3 Current (Derived)']
colors = ['Reds', 'Greens', 'Blues']

for i in range(3):
    # 정규화된 2D 배열의 상별 시각화 (vmin=0, vmax=1 고정)
    mesh = axes[i].pcolormesh(t, f, stft_image[:, :, i], shading='gouraud', cmap=colors[i], vmin=0, vmax=1)
    axes[i].set_ylabel('Frequency [Hz]')
    axes[i].set_title(f'Normalized STFT Spectrogram - {channel_names[i]}')
    
    # 64kHz 나이퀴스트 주파수는 32000Hz지만, 모터 고장 패턴은 주로 5000Hz 이하 대역에서 관찰됩니다.
    axes[i].set_ylim([0, 5000]) 
    fig.colorbar(mesh, ax=axes[i], format='%.2f')

axes[-1].set_xlabel('Time [sec]')
plt.tight_layout()
plt.show()\n